# Bash/Python glue patterns: parsing, looping, error handling

> L2 concept exercise — I wrote this notebook to practice the three glue patterns I reach for most when a script needs to drive other programs: parsing command output, looping over results, and handling failures cleanly. Each section starts from a real-ish problem and shows the bash side and the Python side of the same idea.


## Setup

Everything below runs on plain Python 3 with the standard library — no pip installs. When I need to call bash, I use `subprocess` and capture both stdout and stderr so I can inspect what actually happened instead of guessing.


In [ ]:
# last_verified: 2026-08-11 · scripting & automation (bash/python) n/a
import subprocess
import time


def bash(cmd: str) -> subprocess.CompletedProcess:
    """Run a bash command and return the full result so I can inspect rc/stdout/stderr."""
    return subprocess.run(["bash", "-c", cmd], capture_output=True, text=True)

## Pattern 1 — parsing command output

Problem: a tool prints a multi-line table and I need one column of it as a list. On a real box this could be `ps`, `df`, or `kubectl get pods`. I faked a small table here so the notebook runs anywhere.

In bash the go-to is `awk`; in Python I split lines and fields myself, which makes the column choice explicit.

In [ ]:
# The "tool output" I'm pretending came from some command
fake = """NAME  STATUS   NODE
api   Running  node-1
db    Pending  node-2
web   Running  node-1
"""

# bash equivalent: feed the text in and let awk pull the first column
res = subprocess.run(
    ["bash", "-c", "awk 'NR>1 {print $1}'"],
    input=fake,
    capture_output=True,
    text=True,
)
print("awk names:", res.stdout.split())

# Python equivalent: split on newlines, skip the header, take the first field
lines = [ln.split() for ln in fake.strip().splitlines()]
names = [fields[0] for fields in lines[1:]]
print("python names:", names)

The bash one-liner wins when I'm already in a terminal. The Python version wins when I also want the other columns — I can keep the whole row as a structure instead of throwing it away.

## Pattern 2 — looping over results

Problem: I have a list of names and need to run one command per name, collecting successes and failures so I can report at the end. This is the "glue" part — loop in the outer language, command in the inner one.


In [ ]:
# I'll pretend to restart a few services. `true` simulates a clean restart.
services = ["api", "db", "web"]

results = {}
for name in services:
    r = bash(f"echo restarting {name} && true")
    results[name] = "ok" if r.returncode == 0 else "failed"

for name, status in results.items():
    print(f"{name}: {status}")

assert all(s == "ok" for s in results.values()), "all services should restart cleanly"

Two details I keep forgetting: always capture the return code per iteration (one failure shouldn't abort the loop), and report the full picture after the loop rather than printing inside it. When a command legitimately fails, I mark it and move on.

## Pattern 3 — error handling with retry

Problem: a flaky command sometimes fails on the first try. Rather than give up, I retry a few times with a short backoff, and only surface the error if every attempt fails.


In [ ]:
# Simulate a command that fails twice then succeeds
attempts_used = {"n": 0}


def flaky() -> int:
    attempts_used["n"] += 1
    return 1 if attempts_used["n"] < 3 else 0


def run_with_retry(cmd_callable, max_tries=3, delay=0.2):
    for attempt in range(1, max_tries + 1):
        rc = cmd_callable()
        if rc == 0:
            return attempt
        if attempt < max_tries:
            time.sleep(delay)  # simple backoff; I'd grow this on a real flaky network
    raise RuntimeError(f"command failed after {max_tries} tries")


ok_on = run_with_retry(flaky)
print(f"succeeded on attempt {ok_on}")

try:
    run_with_retry(lambda: 1, max_tries=2)  # always fails
except RuntimeError as e:
    print("handled:", e)

The retry wrapper is the pattern I reuse everywhere: bounded attempts, short delay, and a loud exception only after the last try. The same shape in bash uses a loop with `-lt $MAX` and a counter — identical logic, just different syntax.

## Verify

After running every cell top to bottom I check that:

1. The parse section printed the same names from both `awk` and the Python split — `api`, `db`, `web`.
2. The loop section printed `ok` for all three services and the assert passed.
3. The retry section reported `succeeded on attempt 3`, then a handled `RuntimeError`.

If any cell errored, the likely cause is that `bash` isn't on `PATH` for the subprocess — it's a Linux-only pattern by design.